
# Pipeline  Traitement de la variable REVENU

Ce script met en place un **pipeline complet et optimisé** pour analyser et contrôler la qualité de la variable de revenu (`MONTANT NET`) dans le panel de solde, avec un traitement pensé pour les **grandes volumétries** (par chunks).

Il comprend :

- une classe de **contrôle qualité** (`ControleQualite`),
- des fonctions de **chargement** depuis S3/MinIO,
- des fonctions d’**exploration descriptive**,
- des fonctions de **détection d’erreurs non temporelles et temporelles**,
- un **pipeline principal** `pipeline_revenu_ultra_optimise()` qui enchaîne toutes les étapes.

---

## 1. Classe `ControleQualite`

```python
class ControleQualite:
    ...
````

### Rôle

* Centraliser tous les **résultats de contrôle**,
* afficher les statistiques d’erreurs par flag,
* exporter des **échantillons de cas problématiques**,
* générer un **rapport global** synthétique.

### Attributs

* `output_dir` : répertoire où sont stockés les fichiers de contrôle (`.csv`, `rapport_global.txt`),
* `resultats` : dictionnaire qui regroupe les résultats par variable.

### Méthodes principales

#### `__init__(output_dir="controles_qualite")`

* Crée le dossier de sortie si nécessaire (`os.makedirs`),
* initialise la structure `resultats`,
* affiche un message de confirmation.

#### `controler_variable(df, variable_name, flags_config)`

* Prend :

  * `df` : DataFrame complet,
  * `variable_name` : nom de la variable contrôlée (ex. `"MONTANT NET"`),
  * `flags_config` : dictionnaire de configuration des flags, de la forme :

    ```python
    {
      "FLAG_X": {
        "description": "...",
        "colonnes_export": [ ... ]
      },
      ...
    }
    ```
* Pour chaque flag présent dans `df` :

  * compte le nombre de lignes avec flag = 1,
  * calcule le pourcentage d’observations concernées,
  * enregistre ces infos dans `self.resultats[variable_name]["flags"]`,
  * affiche un résumé dans la console,
  * si des erreurs existent :

    * appelle `_exporter_cas_problematiques(...)` pour produire un échantillon `.csv`.

#### `_exporter_cas_problematiques(df_erreurs, flag_name, colonnes, limite=500)`

* Prépare une **extraction ciblée** des lignes problématiques :

  * part de `df_erreurs` (lignes où le flag vaut 1),
  * ajoute si disponibles les colonnes de contexte `DATE_COLLECTE` et `PERIODE`,
  * filtre la liste aux colonnes existantes,
  * exporte les `limite` premières lignes (par défaut 500) dans :

    * `output_dir / f"{flag_name}_cas.csv"`.

#### `generer_rapport_global()`

* Construit un fichier texte unique :

  * `output_dir / "rapport_global.txt"`.
* Contenu :

  * en-tête et date,
  * pour chaque variable contrôlée :

    * nombre total d’observations,
    * pour chaque flag :

      * icône (`✅` si 0 erreur, `🔴` sinon),
      * description,
      * nombre d’erreurs + pourcentage,
    * total d’erreurs tous flags confondus + pourcentage.

---

## 2. Fonctions utilitaires – Chargement des données

### `get_s3_client()`

* Crée un client S3/MinIO (`boto3.client`) configuré avec :

  * `endpoint_url` (URL du MinIO),
  * `aws_access_key_id` / `aws_secret_access_key`,
  * `verify=False` (désactive la vérification SSL).

### `charger_panel_solde_1_2()`

* Lit un fichier Parquet depuis le bucket S3/MinIO :

  * `bucket_name = "admindataanstat"`,
  * `object_key = "Solde/Panel_solde_1_2_2015_2020.parquet"`.
* Charge le contenu dans un `DataFrame` via `pd.read_parquet(...)`.
* Affiche :

  * nombre d’observations,
  * nombre de matricules (`MATRICULE`),
  * période des dates de collecte (`DATE_COLLECTE.min() → max()`), si colonnes présentes.
* Retourne le `DataFrame`.

---

## 3. Exploration de la variable REVENU

### `explorer_variable_revenu(df, montant_col="MONTANT NET")`

Réalise une **exploration descriptive rapide** de la variable de revenu.

**Étapes :**

1. Vérifie la présence de la colonne `montant_col` dans `df`.

   * Si absente → message d’erreur et sortie.

2. Conversion en numérique :

   ```python
   df[montant_col] = pd.to_numeric(df[montant_col], errors="coerce")
   ```

3. Informations générales :

   * nombre total d’observations,
   * nombre de valeurs non nulles,
   * nombre et pourcentage de valeurs manquantes.

4. Statistiques descriptives (sur la série non nulle) :

   * moyenne,
   * médiane,
   * écart-type,
   * min,
   * max.

5. Distribution simple :

   * nombre de valeurs négatives,
   * nombre de zéros,
   * nombre de valeurs positives.

6. Quantiles :

   * calcule et affiche les quantiles : 1 %, 5 %, 25 %, 50 %, 75 %, 95 %, 99 %.

---

## 4. Détection des erreurs non temporelles

### `detecter_erreurs_non_temporelles(df, montant_col="MONTANT NET")`

Détecte les **anomalies dans la variable de revenu** sans tenir compte de la dimension temporelle.

**Étapes :**

1. Conversion en numérique :

   ```python
   df[montant_col] = pd.to_numeric(df[montant_col], errors="coerce")
   ```

2. Flags créés :

#### a) `FLAG_MONTANT_MANQUANT`

* `1` si `montant_col` est manquant (`NaN`), `0` sinon.

#### b) `FLAG_MONTANT_TRES_BAS` et `FLAG_MONTANT_TRES_ELEVE`

* Calcule les quantiles sur la série non nulle :

  * `q_low = quantile(1%)`,
  * `q_high = quantile(99,5%)`.
* `FLAG_MONTANT_TRES_BAS` :

  * `1` si montant > 0 et montant < `q_low`.
* `FLAG_MONTANT_TRES_ELEVE` :

  * `1` si montant > `q_high`.
* Si la série est vide → flags mis à 0.

#### c) `FLAG_DOUBLON_STRICT`

* Si `MATRICULE` existe :

  * si `DATE_REF` absent mais `DATE_COLLECTE` présent → crée `DATE_REF` à partir de `DATE_COLLECTE`.
  * si `DATE_REF` existe, crée :

    ```python
    df["FLAG_DOUBLON_STRICT"] = df.duplicated(
        subset=["MATRICULE", "DATE_REF"], keep=False
    ).astype("Int8")
    ```
  * sinon : flag mis à 0.
* Si `MATRICULE` n’existe pas → flag mis à 0.

#### d) `FLAG_MONTANT_NEGATIF`

* `1` si `montant_col < 0`, `0` sinon.

#### e) `FLAG_MONTANT_ZERO`

* `1` si `montant_col == 0`, `0` sinon.

#### f) `FLAG_MONTANT_NET_SUP_BRUT`

* Si `"MONTANT BRUT_B1"` existe :

  * convertit `"MONTANT BRUT_B1"` en numérique,
  * `1` si :

    * `montant_col` non nul,
    * `"MONTANT BRUT_B1"` non nul,
    * `montant_col > MONTANT BRUT_B1`.
* Sinon → flag mis à 0.

**Résultat :**

* Retourne `df` enrichi de tous ces flags.

---

## 5. Détection des erreurs temporelles – Traitement par chunks

### `detecter_erreurs_temporelles_par_chunks(df, montant_col="MONTANT NET", chunk_size=100000)`

Détecte des **variations temporelles anormales** du revenu, en évitant les tris globaux coûteux grâce à un traitement par **chunks de matricules**.

**Pré-requis :**

* `MATRICULE` doit exister, sinon :

  * les flags temporels sont créés à 0 et la fonction sort.
* `DATE_REF` doit exister, sinon :

  * si `DATE_COLLECTE` existe → création de `DATE_REF`,
  * s’il n’y a ni `DATE_REF` ni `DATE_COLLECTE` → flags mis à 0.

**Initialisation :**

* Crée deux colonnes :

  * `FLAG_MONTANT_BAISSE_FORTE = 0`,
  * `FLAG_MONTANT_HAUSSE_FORTE = 0`.

**Traitement par chunks :**

1. Récupère la liste des `MATRICULE` uniques et leur nombre.

2. Boucle par paquets de `chunk_size` matricules :

   * extrait `df_chunk = df[df["MATRICULE"].isin(chunk_matricules)].copy()`,

   * trie `df_chunk` par `["MATRICULE", "DATE_REF"]`,

   * calcule le montant précédent par matricule :

     ```python
     df_chunk["_MONTANT_PREV"] = df_chunk.groupby("MATRICULE")[montant_col].shift(1)
     ```

   * définit les conditions :

     * **Baisse forte** :

       ```python
       baisse_forte = (
           df_chunk[montant_col].notna() &
           df_chunk["_MONTANT_PREV"].notna() &
           (df_chunk[montant_col] < 0.5 * df_chunk["_MONTANT_PREV"])
       )
       ```

     * **Hausse forte** :

       ```python
       hausse_forte = (
           df_chunk[montant_col].notna() &
           df_chunk["_MONTANT_PREV"].notna() &
           (df_chunk[montant_col] > 2.0 * df_chunk["_MONTANT_PREV"])
       )
       ```

   * reporte les flags dans le `DataFrame` global :

     ```python
     df.loc[df_chunk.index, "FLAG_MONTANT_BAISSE_FORTE"] = baisse_forte.astype("Int8")
     df.loc[df_chunk.index, "FLAG_MONTANT_HAUSSE_FORTE"] = hausse_forte.astype("Int8")
     ```

   * libère la mémoire (`del df_chunk` + `gc.collect()`).

3. Affiche une progression conditionnelle sur le nombre de matricules traités.

**Résultat :**

* `df` enrichi avec :

  * `FLAG_MONTANT_BAISSE_FORTE` (division par ≥ 2),
  * `FLAG_MONTANT_HAUSSE_FORTE` (multiplication par ≥ 2).

---

## 6. Pipeline global : `pipeline_revenu_ultra_optimise()`

Ce pipeline enchaîne toutes les étapes précédentes, **de bout en bout**.

### Étapes du pipeline

1. **Chargement des données**

   * Appel à `charger_panel_solde_1_2()`,
   * `df` est chargé depuis S3/MinIO.

2. **Exploration**

   * `explorer_variable_revenu(df, "MONTANT NET")` :

     * diagnostics descriptifs sur `MONTANT NET`.

3. **Erreurs non temporelles**

   * `df = detecter_erreurs_non_temporelles(df, "MONTANT NET")` :

     * création des flags :

       * `FLAG_MONTANT_MANQUANT`,
       * `FLAG_MONTANT_TRES_BAS`,
       * `FLAG_MONTANT_TRES_ELEVE`,
       * `FLAG_DOUBLON_STRICT`,
       * `FLAG_MONTANT_NEGATIF`,
       * `FLAG_MONTANT_ZERO`,
       * `FLAG_MONTANT_NET_SUP_BRUT`.

4. **Erreurs temporelles**

   * `df = detecter_erreurs_temporelles_par_chunks(df, "MONTANT NET", chunk_size=50000)` :

     * création des flags :

       * `FLAG_MONTANT_BAISSE_FORTE`,
       * `FLAG_MONTANT_HAUSSE_FORTE`.

5. **Contrôle qualité**

   * Instanciation de `ControleQualite(output_dir="controles_qualite_revenu")`.

   * Appel à `controle.controler_variable(...)` sur `"MONTANT NET"` avec la configuration des flags :

     * `FLAG_MONTANT_MANQUANT` : valeurs manquantes,
     * `FLAG_MONTANT_TRES_BAS` : valeurs aberrantes très faibles,
     * `FLAG_MONTANT_TRES_ELEVE` : valeurs aberrantes très élevées,
     * `FLAG_DOUBLON_STRICT` : doublons (MATRICULE × DATE_REF),
     * `FLAG_MONTANT_NEGATIF` : montants négatifs,
     * `FLAG_MONTANT_ZERO` : montants nuls,
     * `FLAG_MONTANT_NET_SUP_BRUT` : net > brut,
     * `FLAG_MONTANT_BAISSE_FORTE` : forte baisse,
     * `FLAG_MONTANT_HAUSSE_FORTE` : forte hausse.

   * Pour chaque flag, le système :

     * affiche nb d’erreurs + %,
     * exporte un échantillon `.csv` des cas problématiques.

6. **Rapport global**

   * `controle.generer_rapport_global()` :

     * produit `rapport_global.txt` avec la synthèse de tous les flags.

7. **Retour**

   * Retourne :

     * `df_final` : `df` enrichi de tous les flags,
     * `controle` : instance de `ControleQualite` contenant tous les résultats.

---

## 7. Point d’entrée du script

```python
if __name__ == "__main__":
    df_final, controle = pipeline_revenu_ultra_optimise()
```

* Permet d’exécuter tout le pipeline en mode script :

  * chargement,
  * exploration,
  * détection des erreurs,
  * contrôle qualité,
  * rapport global.

---

## Résumé des principaux flags REVENU

* **Non temporels :**

  * `FLAG_MONTANT_MANQUANT` : revenu manquant,
  * `FLAG_MONTANT_TRES_BAS` : revenu positif mais < quantile 1 %,
  * `FLAG_MONTANT_TRES_ELEVE` : revenu > quantile 99,5 %,
  * `FLAG_DOUBLON_STRICT` : doublons (MATRICULE, DATE_REF),
  * `FLAG_MONTANT_NEGATIF` : revenu négatif,
  * `FLAG_MONTANT_ZERO` : revenu nul,
  * `FLAG_MONTANT_NET_SUP_BRUT` : net > brut.

* **Temporels :**

  * `FLAG_MONTANT_BAISSE_FORTE` : revenu < 50 % du revenu précédent,
  * `FLAG_MONTANT_HAUSSE_FORTE` : revenu > 200 % du revenu précédent.


```
```


In [1]:
"""
Script ULTRA-LÉGER pour le traitement de la variable REVENU
Option : échantillonnage pour les analyses temporelles lourdes
"""

import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import io
import os
import warnings
import gc
warnings.filterwarnings("ignore")


# ============================================================================
# CLASSE DE CONTRÔLE QUALITÉ
# ============================================================================

class ControleQualite:
    """Classe pour gérer les contrôles qualité"""
    
    def __init__(self, output_dir="controles_qualite"):
        self.output_dir = output_dir
        self.resultats = {}
        os.makedirs(output_dir, exist_ok=True)
        print(f"✓ Système de contrôle qualité initialisé")
        print(f"  Répertoire : {output_dir}/\n")
    
    def controler_variable(self, df, variable_name, flags_config):
        """Contrôle une variable selon les flags définis"""
        print(f"\n📋 Contrôle de la variable : {variable_name}")
        print("-" * 60)
        
        resultats_variable = {
            "variable": variable_name,
            "total_observations": len(df),
            "flags": {}
        }
        
        for flag_name, config in flags_config.items():
            if flag_name not in df.columns:
                continue
            
            nb_erreurs = int(df[flag_name].sum())
            pct_erreurs = (nb_erreurs / len(df)) * 100
            
            resultats_variable["flags"][flag_name] = {
                "description": config["description"],
                "nombre": nb_erreurs,
                "pourcentage": round(pct_erreurs, 2)
            }
            
            # Affichage
            if nb_erreurs > 0:
                print(f"  🔴 {flag_name}: {nb_erreurs:,} ({pct_erreurs:.2f}%)")
                print(f"      → {config['description']}")
                
                # Export selon le type de flag
                if flag_name in ["FLAG_MONTANT_BAISSE_FORTE", "FLAG_MONTANT_HAUSSE_FORTE"]:
                    # Export matriciel pour flags temporels
                    self._exporter_flags_temporels_matriciel(
                        df,
                        flag_name,
                        config.get("colonnes_export", [variable_name])
                    )
                else:
                    # Export standard pour autres flags
                    self._exporter_cas_problematiques(
                        df[df[flag_name] == 1],
                        flag_name,
                        config.get("colonnes_export", [variable_name]),
                        limite=500
                    )
            else:
                print(f"  ✅ {flag_name}: Aucune erreur")
        
        self.resultats[variable_name] = resultats_variable
        print()
    
    def _exporter_cas_problematiques(self, df_erreurs, flag_name, colonnes, limite=500):
        """Export les cas problématiques"""
        if len(df_erreurs) == 0:
            return
        
        colonnes_export = list(colonnes)
        for col in ["DATE_COLLECTE", "PERIODE"]:
            if col in df_erreurs.columns and col not in colonnes_export:
                colonnes_export.append(col)
        
        colonnes_export = [c for c in colonnes_export if c in df_erreurs.columns]
        
        df_export = df_erreurs[colonnes_export].head(limite).copy()
        if "MATRICULE" in df_export.columns:
            df_export["MATRICULE"] = "'" + df_export["MATRICULE"].astype(str)
        
        fichier = os.path.join(self.output_dir, f"{flag_name}_cas.csv")
        df_export.to_csv(fichier, index=False)
        print(f"      📁 {fichier} ({min(len(df_erreurs), limite):,} cas)")
    
    def _exporter_flags_temporels_matriciel(self, df_complet, flag_name, colonnes):
        """Export matriciel pour les flags temporels"""
        df_erreurs = df_complet[df_complet[flag_name] == 1].copy()
        
        if len(df_erreurs) == 0:
            return
        
        print(f"      → Génération de l'export matriciel amélioré...")
        
        if "DATE_REF" not in df_complet.columns:
            if "DATE_COLLECTE" in df_complet.columns:
                df_complet["DATE_REF"] = pd.to_datetime(df_complet["DATE_COLLECTE"], errors="coerce")
        
        matricules_concernes = df_erreurs["MATRICULE"].unique()[:200]
        
        print(f"      → Analyse de {len(matricules_concernes):,} matricules...")
        
        resultats = []
        
        for idx, matricule in enumerate(matricules_concernes):
            if idx % 50 == 0 and idx > 0:
                print(f"         Traitement : {idx}/{len(matricules_concernes)}")
                gc.collect()
            
            df_mat = df_complet[df_complet["MATRICULE"] == matricule].copy()
            df_mat = df_mat.sort_values("DATE_REF")
            
            ligne = {
                "MATRICULE": str(matricule),
                "NB_VARIATIONS": int(df_mat[flag_name].sum()),
            }
            
            dates_variations = df_mat[df_mat[flag_name] == 1]["DATE_REF"].tolist()
            
            intervalles_mois = []
            if len(dates_variations) > 1:
                for i in range(1, len(dates_variations)):
                    date1 = dates_variations[i-1]
                    date2 = dates_variations[i]
                    diff_mois = (date2.year - date1.year) * 12 + (date2.month - date1.month)
                    intervalles_mois.append(diff_mois)
                
                ligne["INTERVALLES_MOIS"] = ", ".join(map(str, intervalles_mois))
                ligne["INTERVALLE_MOYEN_MOIS"] = round(np.mean(intervalles_mois), 1)
                ligne["INTERVALLE_MIN_MOIS"] = min(intervalles_mois)
                ligne["INTERVALLE_MAX_MOIS"] = max(intervalles_mois)
            else:
                ligne["INTERVALLES_MOIS"] = "N/A"
                ligne["INTERVALLE_MOYEN_MOIS"] = ""
                ligne["INTERVALLE_MIN_MOIS"] = ""
                ligne["INTERVALLE_MAX_MOIS"] = ""
            
            df_mat["ANNEE"] = df_mat["DATE_REF"].dt.year
            df_mat["MOIS"] = df_mat["DATE_REF"].dt.month
            
            # Filtrer les années valides (pas NaN)
            annees_valides = df_mat["ANNEE"].dropna().unique()
            
            for annee in sorted(annees_valides):
                df_annee = df_mat[df_mat["ANNEE"] == annee]
                montants = df_annee["MONTANT NET"].values
                if len(montants) > 0:
                    ligne[f"MONTANT_{int(annee)}"] = f"{montants[0]:,.0f}"
                
                if df_annee[flag_name].sum() > 0:
                    ligne[f"FLAG_{int(annee)}"] = "🔴"
                    mois_variation = df_annee[df_annee[flag_name] == 1]["MOIS"].dropna().values
                    if len(mois_variation) > 0:
                        ligne[f"MOIS_{int(annee)}"] = f"M{int(mois_variation[0])}"
                else:
                    ligne[f"FLAG_{int(annee)}"] = ""
                    ligne[f"MOIS_{int(annee)}"] = ""
            
            resultats.append(ligne)
        
        df_matriciel = pd.DataFrame(resultats)
        
        cols_base = ["MATRICULE", "NB_VARIATIONS", "INTERVALLES_MOIS", 
                     "INTERVALLE_MOYEN_MOIS", "INTERVALLE_MIN_MOIS", "INTERVALLE_MAX_MOIS"]
        cols_annees = [col for col in df_matriciel.columns if col not in cols_base]
        df_matriciel = df_matriciel[cols_base + sorted(cols_annees)]
        
        fichier = os.path.join(self.output_dir, f"{flag_name}_matriciel.csv")
        df_matriciel.to_csv(fichier, index=False, quoting=1)
        print(f"      📁 {fichier} ({len(df_matriciel):,} matricules)")
        
        fichier_detail = os.path.join(self.output_dir, f"{flag_name}_cas_detail.csv")
        cols_export = ["MATRICULE", "MONTANT NET", "DATE_COLLECTE", "PERIODE"]
        cols_export = [c for c in cols_export if c in df_erreurs.columns]
        
        df_detail = df_erreurs[cols_export].head(500).copy()
        if "MATRICULE" in df_detail.columns:
            df_detail["MATRICULE"] = "'" + df_detail["MATRICULE"].astype(str)
        
        df_detail.to_csv(fichier_detail, index=False)
        print(f"      📁 {fichier_detail} (500 cas détaillés)")
    
    def generer_rapport_global(self):
        """Génère un rapport global"""
        print("\n" + "=" * 80)
        print("  RAPPORT GLOBAL")
        print("=" * 80 + "\n")
        
        fichier = os.path.join(self.output_dir, "rapport_global.txt")
        
        with open(fichier, "w", encoding="utf-8") as f:
            f.write("RAPPORT GLOBAL DE CONTRÔLE QUALITÉ\n")
            f.write("=" * 80 + "\n\n")
            f.write(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            for var_name, resultats in self.resultats.items():
                f.write(f"\nVariable : {var_name}\n")
                f.write(f"Total observations : {resultats['total_observations']:,}\n\n")
                
                total_erreurs = 0
                for flag_name, flag_data in resultats["flags"].items():
                    nb = flag_data["nombre"]
                    pct = flag_data["pourcentage"]
                    desc = flag_data["description"]
                    
                    status = "✅" if nb == 0 else "🔴"
                    f.write(f"{status} {flag_name}\n")
                    f.write(f"   {desc}\n")
                    f.write(f"   {nb:,} ({pct:.2f}%)\n\n")
                    
                    total_erreurs += nb
                
                pct_total = (total_erreurs / resultats['total_observations']) * 100
                f.write(f"\nTotal erreurs : {total_erreurs:,} ({pct_total:.2f}%)\n")
        
        print(f"✅ Rapport généré : {fichier}\n")


# ============================================================================
# FONCTIONS UTILITAIRES
# ============================================================================

def get_s3_client():
    """Client S3/MinIO"""
    return boto3.client(
        "s3",
        endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False
    )


def charger_panel_solde_1_2():
    """Charge les données depuis S3/MinIO"""
    bucket_name = "admindataanstat"
    object_key = "Solde/Panel_solde_1_2_2015_2020.parquet"

    s3_client = get_s3_client()
    obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    
    print(f"✓ Données chargées : {len(df):,} observations")
    if "MATRICULE" in df.columns:
        print(f"✓ Matricules uniques : {df['MATRICULE'].nunique():,}")
    if "DATE_COLLECTE" in df.columns:
        print(f"✓ Période : {df['DATE_COLLECTE'].min()} → {df['DATE_COLLECTE'].max()}")
    
    return df


# ============================================================================
# VÉRIFICATION DES MATRICULES
# ============================================================================

def verifier_matricules(df):
    """Vérifie la qualité des matricules"""
    print("\n" + "="*80)
    print("  VÉRIFICATION DES MATRICULES")
    print("="*80 + "\n")
    
    if "MATRICULE" not in df.columns:
        print("❌ Colonne MATRICULE absente")
        return df
    
    print("1️⃣  STATISTIQUES GÉNÉRALES")
    print("-" * 60)
    total_matricules = df["MATRICULE"].nunique()
    print(f"   Matricules uniques : {total_matricules:,}")
    print()
    
    print("2️⃣  MATRICULES NUMÉRIQUES")
    print("-" * 60)
    
    df["MATRICULE_STR"] = df["MATRICULE"].astype(str)
    df["FLAG_MATRICULE_NUMERIQUE"] = df["MATRICULE_STR"].str.isdigit().astype("Int8")
    
    nb_numeriques = df["FLAG_MATRICULE_NUMERIQUE"].sum()
    matricules_numeriques = df[df["FLAG_MATRICULE_NUMERIQUE"] == 1]["MATRICULE"].unique()
    
    print(f"   Observations avec matricule numérique : {nb_numeriques:,}")
    print(f"   Matricules uniques numériques : {len(matricules_numeriques):,}")
    
    if len(matricules_numeriques) > 0:
        print(f"\n   Exemples :")
        for i, mat in enumerate(matricules_numeriques[:10], 1):
            nb_obs = (df["MATRICULE"] == mat).sum()
            print(f"      {i}. {mat} ({nb_obs:,} observations)")
    print()
    
    output_dir = "controles_qualite_revenu"
    os.makedirs(output_dir, exist_ok=True)
    
    if nb_numeriques > 0:
        fichier = os.path.join(output_dir, "FLAG_MATRICULE_NUMERIQUE_cas.csv")
        df_num = df[df["FLAG_MATRICULE_NUMERIQUE"] == 1][["MATRICULE", "MONTANT NET", "DATE_COLLECTE"]].head(500).copy()
        df_num["MATRICULE"] = "'" + df_num["MATRICULE"].astype(str)
        df_num.to_csv(fichier, index=False)
        print(f"   📁 {fichier} ({min(nb_numeriques, 500):,} cas)")
    
    print()
    return df


# ============================================================================
# EXPLORATION
# ============================================================================

def explorer_variable_revenu(df, montant_col="MONTANT NET"):
    """Exploration de la variable revenu"""
    print("\n" + "="*80)
    print("  EXPLORATION DE LA VARIABLE REVENU")
    print("="*80 + "\n")
    
    if montant_col not in df.columns:
        print(f"❌ Colonne '{montant_col}' non trouvée")
        return
    
    df[montant_col] = pd.to_numeric(df[montant_col], errors="coerce")
    
    print(f"📊 Variable : {montant_col}\n")
    
    print("1️⃣  INFORMATIONS GÉNÉRALES")
    print("-" * 60)
    total = len(df)
    non_null = df[montant_col].notna().sum()
    null = df[montant_col].isna().sum()
    print(f"   Total observations : {total:,}")
    print(f"   Valeurs non-nulles : {non_null:,}")
    print(f"   Valeurs manquantes : {null:,} ({null/total*100:.2f}%)")
    print()
    
    print("2️⃣  STATISTIQUES DESCRIPTIVES")
    print("-" * 60)
    serie = df[montant_col].dropna()
    if len(serie) > 0:
        print(f"   Moyenne : {serie.mean():,.2f}")
        print(f"   Médiane : {serie.median():,.2f}")
        print(f"   Écart-type : {serie.std():,.2f}")
        print(f"   Minimum : {serie.min():,.2f}")
        print(f"   Maximum : {serie.max():,.2f}")
    print()
    
    print("3️⃣  DISTRIBUTION")
    print("-" * 60)
    negatifs = (df[montant_col] < 0).sum()
    zeros = (df[montant_col] == 0).sum()
    positifs = (df[montant_col] > 0).sum()
    print(f"   Valeurs négatives : {negatifs:,}")
    print(f"   Valeurs nulles : {zeros:,}")
    print(f"   Valeurs positives : {positifs:,}")
    print()


# ============================================================================
# DÉTECTION DES ERREURS
# ============================================================================

def detecter_erreurs_non_temporelles(df, montant_col="MONTANT NET"):
    """Détecte les erreurs NON temporelles"""
    print("\n🔍 Détection des erreurs non temporelles...")
    
    df[montant_col] = pd.to_numeric(df[montant_col], errors="coerce")
    
    print("   → Valeurs manquantes ponctuelles...")
    df["FLAG_MONTANT_MANQUANT"] = df[montant_col].isna().astype("Int8")
    
    print("   → Années complètes manquantes par matricule...")
    df = detecter_annees_manquantes(df, montant_col)
    
    print("   → Valeurs aberrantes...")
    serie = df[montant_col].dropna()
    if len(serie) > 0:
        q_low = serie.quantile(0.01)
        q_high = serie.quantile(0.995)
        df["FLAG_MONTANT_TRES_BAS"] = (
            (df[montant_col] > 0) & (df[montant_col] < q_low)
        ).astype("Int8")
        df["FLAG_MONTANT_TRES_ELEVE"] = (
            df[montant_col] > q_high
        ).astype("Int8")
    else:
        df["FLAG_MONTANT_TRES_BAS"] = 0
        df["FLAG_MONTANT_TRES_ELEVE"] = 0
    
    print("   → Doublons...")
    if "MATRICULE" in df.columns:
        if "DATE_REF" not in df.columns:
            if "DATE_COLLECTE" in df.columns:
                df["DATE_REF"] = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce")
        
        if "DATE_REF" in df.columns:
            df["FLAG_DOUBLON_STRICT"] = df.duplicated(
                subset=["MATRICULE", "DATE_REF"], keep=False
            ).astype("Int8")
        else:
            df["FLAG_DOUBLON_STRICT"] = 0
    else:
        df["FLAG_DOUBLON_STRICT"] = 0
    
    print("   → Données erronées...")
    df["FLAG_MONTANT_NEGATIF"] = (df[montant_col] < 0).astype("Int8")
    df["FLAG_MONTANT_ZERO"] = (df[montant_col] == 0).astype("Int8")
    
    print("   → Incohérence net/brut...")
    if "MONTANT BRUT_B1" in df.columns:
        df["MONTANT BRUT_B1"] = pd.to_numeric(df["MONTANT BRUT_B1"], errors="coerce")
        df["FLAG_MONTANT_NET_SUP_BRUT"] = (
            df[montant_col].notna() &
            df["MONTANT BRUT_B1"].notna() &
            (df[montant_col] > df["MONTANT BRUT_B1"])
        ).astype("Int8")
    else:
        df["FLAG_MONTANT_NET_SUP_BRUT"] = 0
    
    print("✅ Erreurs non temporelles détectées\n")
    return df


def detecter_annees_manquantes(df, montant_col="MONTANT NET"):
    """
    Détecte les années complètes manquantes pour chaque matricule
    Exemple : Un matricule présent en 2015, 2016, 2018, 2019 → 2017 manquante
    """
    if "DATE_REF" not in df.columns:
        if "DATE_COLLECTE" in df.columns:
            df["DATE_REF"] = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce")
        else:
            df["FLAG_ANNEE_MANQUANTE"] = 0
            return df
    
    if "MATRICULE" not in df.columns:
        df["FLAG_ANNEE_MANQUANTE"] = 0
        return df
    
    # Extraire l'année
    df["_ANNEE"] = df["DATE_REF"].dt.year
    
    # Initialiser le flag
    df["FLAG_ANNEE_MANQUANTE"] = 0
    
    # Identifier les années attendues (2015-2020)
    annees_attendues = set(range(2015, 2021))
    
    # Pour chaque matricule, identifier les années manquantes
    matricules_avec_annees_manquantes = []
    
    # Analyser par matricule (échantillon pour performance)
    matricules_uniques = df["MATRICULE"].unique()
    
    for matricule in matricules_uniques[:100000]:  # Limiter pour éviter surcharge
        df_mat = df[df["MATRICULE"] == matricule]
        annees_presentes = set(df_mat["_ANNEE"].dropna().unique())
        
        if len(annees_presentes) > 0:
            # Années où le matricule existe
            annee_min = min(annees_presentes)
            annee_max = max(annees_presentes)
            
            # Années attendues dans cette période
            annees_attendues_matricule = set(range(int(annee_min), int(annee_max) + 1))
            
            # Années manquantes
            annees_manquantes = annees_attendues_matricule - annees_presentes
            
            if len(annees_manquantes) > 0:
                # Marquer toutes les observations de ce matricule
                df.loc[df["MATRICULE"] == matricule, "FLAG_ANNEE_MANQUANTE"] = 1
                matricules_avec_annees_manquantes.append({
                    "MATRICULE": matricule,
                    "ANNEES_PRESENTES": sorted(list(annees_presentes)),
                    "ANNEES_MANQUANTES": sorted(list(annees_manquantes)),
                    "NB_ANNEES_MANQUANTES": len(annees_manquantes)
                })
    
    df["FLAG_ANNEE_MANQUANTE"] = df["FLAG_ANNEE_MANQUANTE"].astype("Int8")
    
    # Exporter le détail des années manquantes
    if len(matricules_avec_annees_manquantes) > 0:
        output_dir = "controles_qualite_revenu"
        os.makedirs(output_dir, exist_ok=True)
        
        df_annees_manquantes = pd.DataFrame(matricules_avec_annees_manquantes)
        df_annees_manquantes["MATRICULE"] = "'" + df_annees_manquantes["MATRICULE"].astype(str)
        
        fichier = os.path.join(output_dir, "ANNEES_MANQUANTES_detail.csv")
        df_annees_manquantes.to_csv(fichier, index=False)
        print(f"      📁 Détail des années manquantes : {fichier}")
    
    # Nettoyer
    df = df.drop(columns=["_ANNEE"], errors="ignore")
    
    return df


def detecter_erreurs_temporelles_leger(df, montant_col="MONTANT NET", sample_size=50000):
    """
    Version ULTRA-LÉGÈRE : Analyse sur échantillon uniquement
    Évite le crash en traitant moins de données
    """
    print("\n🔍 Détection des erreurs temporelles (VERSION LÉGÈRE - ÉCHANTILLON)")
    print("   ⚠️  Analyse sur échantillon pour éviter les problèmes de mémoire")
    print()
    
    if "MATRICULE" not in df.columns:
        print("   ⚠️ Colonne MATRICULE absente - étape ignorée")
        df["FLAG_MONTANT_BAISSE_FORTE"] = 0
        df["FLAG_MONTANT_HAUSSE_FORTE"] = 0
        return df
    
    if "DATE_REF" not in df.columns:
        if "DATE_COLLECTE" in df.columns:
            df["DATE_REF"] = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce")
        else:
            print("   ⚠️ Colonne DATE_REF absente - étape ignorée")
            df["FLAG_MONTANT_BAISSE_FORTE"] = 0
            df["FLAG_MONTANT_HAUSSE_FORTE"] = 0
            return df
    
    # Initialiser les flags à 0 pour tout le monde
    df["FLAG_MONTANT_BAISSE_FORTE"] = 0
    df["FLAG_MONTANT_HAUSSE_FORTE"] = 0
    
    # Échantillonner les matricules
    all_matricules = df["MATRICULE"].unique()
    nb_total = len(all_matricules)
    
    # Prendre un échantillon aléatoire
    nb_sample = min(sample_size, nb_total)
    matricules_sample = np.random.choice(all_matricules, size=nb_sample, replace=False)
    
    print(f"   → Échantillon : {nb_sample:,} matricules sur {nb_total:,} ({nb_sample/nb_total*100:.1f}%)")
    print(f"   → Traitement en cours...")
    
    # Filtrer sur l'échantillon
    df_sample = df[df["MATRICULE"].isin(matricules_sample)].copy()
    df_sample = df_sample.sort_values(["MATRICULE", "DATE_REF"])
    
    # Calculer les variations
    df_sample["_MONTANT_PREV"] = df_sample.groupby("MATRICULE")[montant_col].shift(1)
    
    baisse_forte = (
        df_sample[montant_col].notna() &
        df_sample["_MONTANT_PREV"].notna() &
        (df_sample[montant_col] < 0.5 * df_sample["_MONTANT_PREV"])
    )
    
    hausse_forte = (
        df_sample[montant_col].notna() &
        df_sample["_MONTANT_PREV"].notna() &
        (df_sample[montant_col] > 2.0 * df_sample["_MONTANT_PREV"])
    )
    
    # Mettre à jour uniquement pour l'échantillon
    df.loc[df_sample.index, "FLAG_MONTANT_BAISSE_FORTE"] = baisse_forte.astype("Int8")
    df.loc[df_sample.index, "FLAG_MONTANT_HAUSSE_FORTE"] = hausse_forte.astype("Int8")
    
    nb_baisses = baisse_forte.sum()
    nb_hausses = hausse_forte.sum()
    
    print(f"   ✅ Traitement terminé")
    print(f"   → Baisses fortes détectées : {nb_baisses:,}")
    print(f"   → Hausses fortes détectées : {nb_hausses:,}")
    print(f"   ℹ️  Note : Extrapolation totale ≈ {int(nb_baisses * nb_total / nb_sample):,} baisses")
    print(f"   ℹ️  Note : Extrapolation totale ≈ {int(nb_hausses * nb_total / nb_sample):,} hausses")
    print()
    
    del df_sample
    gc.collect()
    
    return df


# ============================================================================
# PIPELINE FINAL LÉGER
# ============================================================================

def pipeline_revenu_leger():
    """Pipeline léger avec échantillonnage"""
    print("\n" + "="*80)
    print("  PIPELINE LÉGER - TRAITEMENT VARIABLE REVENU")
    print("  Analyse temporelle sur ÉCHANTILLON (évite crash mémoire)")
    print("="*80 + "\n")
    
    # 1. Chargement
    print("📂 ÉTAPE 1 : Chargement des données\n")
    df = charger_panel_solde_1_2()
    
    # 2. Vérification des matricules
    print("\n🔍 ÉTAPE 2 : Vérification des matricules")
    df = verifier_matricules(df)
    
    # 3. Exploration
    print("\n📊 ÉTAPE 3 : Exploration de la variable revenu")
    explorer_variable_revenu(df, montant_col="MONTANT NET")
    
    # 4. Détection des erreurs NON temporelles
    print("\n🔍 ÉTAPE 4a : Détection des erreurs non temporelles")
    df = detecter_erreurs_non_temporelles(df, montant_col="MONTANT NET")
    
    # 5. Détection des erreurs TEMPORELLES (ÉCHANTILLON)
    print("\n🔍 ÉTAPE 4b : Détection des erreurs temporelles")
    df = detecter_erreurs_temporelles_leger(df, montant_col="MONTANT NET", sample_size=50000)
    
    # 6. Contrôle qualité
    print("\n📋 ÉTAPE 5 : Contrôle qualité")
    controle = ControleQualite(output_dir="controles_qualite_revenu")
    
    controle.controler_variable(
        df,
        "MONTANT NET",
        {
            "FLAG_MONTANT_MANQUANT": {
                "description": "Valeurs manquantes ponctuelles (ligne par ligne)",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_ANNEE_MANQUANTE": {
                "description": "Années complètes manquantes dans la période du matricule",
                "colonnes_export": ["MONTANT NET", "MATRICULE", "DATE_COLLECTE"]
            },
            "FLAG_MONTANT_TRES_BAS": {
                "description": "Valeurs aberrantes - très faibles",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_MONTANT_TRES_ELEVE": {
                "description": "Valeurs aberrantes - très élevées",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_DOUBLON_STRICT": {
                "description": "Doublons",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_MONTANT_NEGATIF": {
                "description": "Données erronées - négatif",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_MONTANT_ZERO": {
                "description": "Données erronées - nul",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_MONTANT_NET_SUP_BRUT": {
                "description": "Incohérence - net > brut",
                "colonnes_export": ["MONTANT NET", "MONTANT BRUT_B1", "MATRICULE"]
            },
            "FLAG_MONTANT_BAISSE_FORTE": {
                "description": "Incohérence temporelle - baisse forte (ÉCHANTILLON)",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            },
            "FLAG_MONTANT_HAUSSE_FORTE": {
                "description": "Incohérence temporelle - hausse forte (ÉCHANTILLON)",
                "colonnes_export": ["MONTANT NET", "MATRICULE"]
            }
        }
    )
    
    # 7. Rapport global
    controle.generer_rapport_global()
    
    print("\n" + "="*80)
    print("  ✅ PIPELINE TERMINÉ")
    print("="*80 + "\n")
    print("💡 IMPORTANT :")
    print("   • Analyse temporelle basée sur un ÉCHANTILLON de 50,000 matricules")
    print("   • Les flags temporels ne couvrent que l'échantillon analysé")
    print("   • Les autres analyses sont complètes sur toutes les données")
    print()
    
    return df, controle


if __name__ == "__main__":
    df_final, controle = pipeline_revenu_leger()


  PIPELINE LÉGER - TRAITEMENT VARIABLE REVENU
  Analyse temporelle sur ÉCHANTILLON (évite crash mémoire)

📂 ÉTAPE 1 : Chargement des données

✓ Données chargées : 15,632,484 observations
✓ Matricules uniques : 308,776
✓ Période : 2015-01-31 00:00:00 → 2020-12-31 00:00:00

🔍 ÉTAPE 2 : Vérification des matricules

  VÉRIFICATION DES MATRICULES

1️⃣  STATISTIQUES GÉNÉRALES
------------------------------------------------------------
   Matricules uniques : 308,776

2️⃣  MATRICULES NUMÉRIQUES
------------------------------------------------------------
   Observations avec matricule numérique : 45
   Matricules uniques numériques : 45

   Exemples :
      1. 02 (1 observations)
      2. 03 (1 observations)
      3. 04 (1 observations)
      4. 05 (1 observations)
      5. 06 (1 observations)
      6. 09 (1 observations)
      7. 12 (1 observations)
      8. 13 (1 observations)
      9. 14 (1 observations)
      10. 15 (1 observations)

   📁 controles_qualite_revenu/FLAG_MATRICULE_NUMERIQU

KeyboardInterrupt: 